# Block 1: Environment Setup, Configuration, and Path Definitions
This block initializes the environment, mounts the storage, and defines the hyperparameters. It establishes the standard montage (22 channels) to ensure spatial consistency between Patient A and Patient B.

In [ ]:
!pip install mne

import numpy as np
import os
import mne
import sys
import warnings
from scipy.signal import butter, filtfilt, welch
from scipy.integrate import simpson
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import average_precision_score, confusion_matrix, roc_curve, classification_report

# Suppress runtime warnings from MNE regarding channel renaming for cleaner output
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ==================== 1. Drive Mounting & Path Configuration ====================
if not os.path.exists('/content/drive'):
    from google.colab import drive
    print("[INFO] Mounting Google Drive...")
    drive.mount('/content/drive')

# Define file paths. Ensure these match your specific directory structure.
# Source Domain (Patient A)
PATH_A_NPZ = "/content/drive/MyDrive/chb01_processed_data.npz"
# Target Domain (Patient B)
DIR_B_RAW  = "/content/drive/MyDrive/chb02"
# Summary File (Ground Truth Labels)
SUMMARY_B  = "/content/drive/MyDrive/chb02/chb02-summary.txt"

# Path validation logic
if not os.path.exists(SUMMARY_B):
    # Fallback check: try the parent directory if not found in subdirectory
    alt_path = "/content/drive/MyDrive/chb02-summary.txt"
    if os.path.exists(alt_path):
        print(f"[WARNING] Summary file not found at primary path. Found at: {alt_path}")
        SUMMARY_B = alt_path
    else:
        raise FileNotFoundError(f"[FATAL] Summary file not found at: {SUMMARY_B}")
else:
    print(f"[INFO] Summary file verified: {SUMMARY_B}")

# ==================== 2. Constants & Hyperparameters ====================
# Standardized Channel Montage (22 Channels)
# This ensures feature vectors are spatially aligned between subjects.
REF_CH_NAMES = ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
                'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
                'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8']

# Frequency bands for Spectral Power Density (PSD) analysis
BANDS = {"Delta": (0.5, 4), "Theta": (4, 8), "Alpha": (8, 13), "Beta": (13, 30)}

# Epoch length in seconds (Analysis window)
EPOCH_LEN_S = 2.0

# Temporal Stacking Window (W)
# We stack features from W consecutive epochs to capture temporal evolution.
W = 3

print("[INFO] Block 1: Environment and configuration initialized successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 20.5 MB/s eta 0:00:00
[INFO] Mounting Google Drive...
Mounted at /content/drive
[INFO] Summary file verified: /content/drive/MyDrive/chb02/chb02-summary.txt
[INFO] Block 1: Environment and configuration initialized successfully.


# Block 2: Signal Processing and Feature Extraction Library
This block defines the mathematical functions for signal conditioning and feature engineering. Key Scientific Features:

Spectral Features: Band power via Welch's method (captures oscillatory activity).

Temporal Features: RMS (amplitude), Line Length (signal complexity/seizure onset detection), and Hjorth Parameters (statistical signal description).

In [ ]:
# ==================== Signal Processing Utilities ====================

def bandpass_filter_multich(data, fs, l_freq, h_freq, order=4):
    """
    Applies a zero-phase Butterworth bandpass filter to multichannel EEG data.
    Zero-phase filtering (filtfilt) is crucial to preserve peak latency.
    """
    nyq = 0.5 * fs
    b, a = butter(order, [l_freq/nyq, h_freq/nyq], btype="band")
    return filtfilt(b, a, data, axis=-1)

def segment_eeg_into_epochs(eeg, sfreq, epoch_len_s):
    """
    Segments continuous EEG signals into non-overlapping epochs.
    Reshapes data to (Epochs, Channels, Samples).
    """
    epoch_len_samples = int(epoch_len_s * sfreq)
    C, N = eeg.shape
    n_epochs = N // epoch_len_samples
    trimmed = eeg[:, : n_epochs * epoch_len_samples]
    epochs = trimmed.reshape(C, n_epochs, epoch_len_samples).transpose(1, 0, 2)
    start_samples = np.arange(n_epochs) * epoch_len_samples
    return {
        "epochs": epochs, "sfreq": sfreq, "epoch_len_s": epoch_len_s,
        "epoch_start_times": start_samples / sfreq,
        "epoch_end_times": (start_samples + epoch_len_samples) / sfreq,
        "n_epochs": n_epochs
    }

def create_labels_from_intervals(seg, seizure_intervals):
    """
    Generates binary labels (0=Interictal, 1=Ictal) based on clinical annotations.
    """
    y = np.zeros(seg["n_epochs"], dtype=int)
    if not seizure_intervals: return y
    for i in range(seg["n_epochs"]):
        t0, t1 = seg["epoch_start_times"][i], seg["epoch_end_times"][i]
        for (s, e) in seizure_intervals:
            if (t0 < e) and (t1 > s): # Check for temporal overlap
                y[i] = 1; break
    return y

def temporal_stack_within_file(X_spatial, y_epoch, W):
    """
    Incorporates temporal context by stacking features from W previous epochs.
    """
    E, F = X_spatial.shape
    if E < W: return np.empty((0, W * F)), np.empty((0,), dtype=int)
    X_final = np.vstack([X_spatial[i-(W-1):i+1].reshape(-1) for i in range(W-1, E)])
    y_final = np.array([y_epoch[i] for i in range(W-1, E)], dtype=int)
    return X_final, y_final

# ==================== Feature Extraction Module ====================

def extract_spatial_features_all_epochs(epochs, fs, bands):
    """
    Extracts a feature vector for each channel in each epoch.
    Features extracted:
    1-4. PSD Band Power (Delta, Theta, Alpha, Beta) via Simpson's rule integration.
    5.   Root Mean Square (RMS) - Measure of signal power.
    6.   Line Length - Measure of signal complexity/waveform dimensionality.
    7.   Hjorth Complexity - Measure of signal similarity to a sine wave.

    Total features per channel: 7.
    """
    E, C, S = epochs.shape
    eps = 1e-12 # Epsilon for numerical stability
    freqs, psd = welch(epochs, fs=fs, nperseg=S, axis=-1)
    feat_list = []

    # Frequency Domain Features
    for (f_low, f_high) in bands.values():
        idx = (freqs >= f_low) & (freqs < f_high)
        be = simpson(psd[..., idx], freqs[idx], axis=-1) if np.any(idx) else np.zeros((E, C))
        feat_list.append(be)

    # Time Domain Features
    # RMS
    feat_list.append(np.sqrt(np.mean(epochs ** 2, axis=-1)))

    # Line Length (Vectorized implementation)
    feat_list.append(np.sum(np.abs(np.diff(epochs, axis=-1)), axis=-1))

    # Hjorth Parameters (Complexity only)
    std_x = np.std(epochs, axis=-1)
    dx = np.diff(epochs, axis=-1); std_dx = np.std(dx, axis=-1)
    ddx = np.diff(dx, axis=-1); std_ddx = np.std(ddx, axis=-1)
    feat_list.append((std_ddx / (std_dx + eps)) / ((std_dx / (std_x + eps)) + eps))

    # Stack features: Shape (Epochs, Channels * 7)
    feat_3d = np.stack(feat_list, axis=-1)
    return feat_3d.reshape(E, C * feat_3d.shape[-1])

print("[INFO] Block 2: Feature extraction functions defined.")

[INFO] Block 2: Feature extraction functions defined.


# Block 3: Loading Source Domain Data (Patient A)
This block loads the pre-computed features for Patient A. It performs a dimensionality check and automatically corrects the feature matrix if an extra channel (duplicate T8-P8) was present in the source data, ensuring the feature space matches the standard 22-channel montage.

In [ ]:
print(f"[INFO] Loading source domain data (Patient A): {PATH_A_NPZ}")
if not os.path.exists(PATH_A_NPZ):
    raise FileNotFoundError("Source data file not found.")

loaded = np.load(PATH_A_NPZ, allow_pickle=True)
X_all_A = loaded['X']
y_all_A = loaded['y']
groups_A = loaded['groups']

print(f"[INFO] Initial Source Data Shape: {X_all_A.shape}")

# === Dimensionality Alignment / Montage Harmonization ===
# Logic: If source data has 483 features (23 channels * 7 features * 3 windows),
# we must remove the redundant channel to match the 22-channel standard (462 features).
if X_all_A.shape[1] == 483:
    print("[INFO] Detected redundant channel in source data (483 dim). Performing dimensionality reduction...")
    N = X_all_A.shape[0]
    # Reshape to [Samples, Window, Channels, Features]
    X_reshaped = X_all_A.reshape(N, 3, 23, 7)

    # Slice to keep only the first 22 channels
    X_fixed = X_reshaped[:, :, :22, :]

    # Flatten back to 2D matrix [Samples, Features]
    X_all_A = X_fixed.reshape(N, -1)
    print(f"[INFO] Alignment complete. New Source Data Shape: {X_all_A.shape}")

# Create Source Training Set (using 70% of Patient A data)
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx_A, _ = next(gss.split(X_all_A, y_all_A, groups_A))
X_train_A, y_train_A = X_all_A[train_idx_A], y_all_A[train_idx_A]

print(f"[INFO] Block 3: Source domain data loaded and aligned. Training samples: {len(y_train_A)}")

[INFO] Loading source domain data (Patient A): /content/drive/MyDrive/chb01_processed_data.npz
[INFO] Initial Source Data Shape: (72909, 483)
[INFO] Detected redundant channel in source data (483 dim). Performing dimensionality reduction...
[INFO] Alignment complete. New Source Data Shape: (72909, 462)
[INFO] Block 3: Source domain data loaded and aligned. Training samples: 52142


# Block 4: Target Domain Processing (Patient B)
This block processes the raw EDF files for Patient B. It includes robust channel label handling (correcting T8-P8-0 to T8-P8) to resolve electrode naming inconsistencies common in clinical EEG datasets.

In [ ]:
print("[INFO] Processing target domain data (Patient B)...")

def parse_summary(summary_path):
    """Parses the clinical summary text file to extract seizure start/end times."""
    seizures = {}
    if not os.path.exists(summary_path): return {}
    with open(summary_path, 'r', encoding='utf-8', errors='ignore') as f:
        current_file = None
        for line in f:
            if line.startswith("File Name:"):
                current_file = line.split(": ")[1].strip()
            elif "Seizure" in line and "Start Time" in line:
                try:
                    start = float(line.split(": ")[1].split()[0])
                    end_line = next(f) # Read the immediate next line for end time
                    end = float(end_line.split(": ")[1].split()[0])
                    seizures.setdefault(current_file, []).append((start, end))
                except Exception as e:
                    print(f"[WARNING] Parsing error in summary file: {e}")
    return seizures

seizures_B = parse_summary(SUMMARY_B)
edf_files_B = sorted([f for f in os.listdir(DIR_B_RAW) if f.endswith('.edf')])

X_list_B, y_list_B = [], []

for i, fname in enumerate(edf_files_B):
    path = os.path.join(DIR_B_RAW, fname)
    try:
        raw = mne.io.read_raw_edf(path, preload=True, verbose=False)

        # --- Channel Label Correction ---
        # Fix specific inconsistency in CHB-MIT dataset where T8-P8 is labeled T8-P8-0
        if 'T8-P8-0' in raw.ch_names and 'T8-P8' not in raw.ch_names:
            mne.rename_channels(raw.info, {'T8-P8-0': 'T8-P8'})

        # Validate Montage
        if not all(ch in raw.ch_names for ch in REF_CH_NAMES):
            # Skip files that do not contain the required channels
            continue

        raw.pick(REF_CH_NAMES)

        # Signal Processing Pipeline
        data = raw.get_data()
        # 1. Bandpass Filter (0.5 - 30 Hz)
        data_bp = bandpass_filter_multich(data, raw.info["sfreq"], 0.5, 30)
        # 2. Segmentation
        seg = segment_eeg_into_epochs(data_bp, raw.info["sfreq"], EPOCH_LEN_S)
        # 3. Label Assignment
        y_ep = create_labels_from_intervals(seg, seizures_B.get(fname, []))

        # 4. Feature Extraction & Temporal Stacking
        X_sp = extract_spatial_features_all_epochs(seg["epochs"], raw.info["sfreq"], BANDS)
        X_fin, y_fin = temporal_stack_within_file(X_sp, y_ep, W)

        if len(y_fin) > 0:
            X_list_B.append(X_fin)
            y_list_B.append(y_fin)
            print(f"[INFO] Processed file {i+1}/{len(edf_files_B)}: {fname}")

        raw.close()
    except Exception as e:
        print(f"[ERROR] Failed to process {fname}: {e}")

if not X_list_B:
    raise ValueError("Target domain processing failed. No data generated.")

X_B_full = np.vstack(X_list_B)
y_B_full = np.concatenate(y_list_B)

print(f"[INFO] Block 4: Target data processing complete. Total Samples: {len(y_B_full)}")
# Verification: X_B_full.shape[1] should now be 462.

[INFO] Processing target domain data (Patient B)...
[INFO] Processed file 1/36: chb02_01.edf
[INFO] Processed file 2/36: chb02_02.edf
[INFO] Processed file 3/36: chb02_03.edf
[INFO] Processed file 4/36: chb02_04.edf
[INFO] Processed file 5/36: chb02_05.edf
[INFO] Processed file 6/36: chb02_06.edf
[INFO] Processed file 7/36: chb02_07.edf
[INFO] Processed file 8/36: chb02_08.edf
[INFO] Processed file 9/36: chb02_09.edf
[INFO] Processed file 10/36: chb02_10.edf
[INFO] Processed file 11/36: chb02_11.edf
[INFO] Processed file 12/36: chb02_12.edf
[INFO] Processed file 13/36: chb02_13.edf
[INFO] Processed file 14/36: chb02_14.edf
[INFO] Processed file 15/36: chb02_15.edf
[INFO] Processed file 16/36: chb02_16+.edf
[INFO] Processed file 17/36: chb02_16.edf
[INFO] Processed file 18/36: chb02_17.edf
[INFO] Processed file 19/36: chb02_18.edf
[INFO] Processed file 20/36: chb02_19.edf
[INFO] Processed file 21/36: chb02_20.edf
[INFO] Processed file 22/36: chb02_21.edf
[INFO] Processed file 23/36: chb

# Block 5: Transfer Learning and Performance Evaluation
This block implements Supervised Domain Adaptation via calibration.

Stratified Split: Ensures the calibration set (20%) contains a representative ratio of seizure samples, preventing the "zero-seizure" testing issue.

Feature Standardization: Re-scales the combined dataset (Patient A + Patient B Calibration) to handle covariate shift (distribution mismatch).

Metrics: Calculates AP, FA/h, and Sensitivity at fixed FPR to provide clinically relevant performance indicators.


In [ ]:
print("[INFO] Initiating Transfer Learning Protocol...")

# 0. Final Dimension Verification
if X_train_A.shape[1] != X_B_full.shape[1]:
    raise ValueError(f"[FATAL] Dimension Mismatch: Source={X_train_A.shape[1]}, Target={X_B_full.shape[1]}")

# 1. Stratified Data Splitting
# We use Stratified Sampling to ensure the calibration set contains seizure examples.
print("[INFO] Performing Stratified Shuffle Split (20% Calibration / 80% Evaluation)...")
X_B_cal, X_B_eval, y_B_cal, y_B_eval = train_test_split(
    X_B_full, y_B_full,
    test_size=0.80,     # 80% reserved for testing
    stratify=y_B_full,  # Maintains class distribution
    random_state=42
)

n_seizures_cal = y_B_cal.sum()
n_seizures_eval = y_B_eval.sum()
print(f"   -> Calibration Set: {len(y_B_cal)} samples ({n_seizures_cal} seizures)")
print(f"   -> Evaluation Set:  {len(y_B_eval)} samples ({n_seizures_eval} seizures)")

if n_seizures_eval == 0:
    raise ValueError("[FATAL] No seizure samples in evaluation set. Check data source.")

# 2. Domain Adaptation (Data Mixing)
# Constructing the training set using Source Domain + Target Calibration Data
X_mix = np.vstack([X_train_A, X_B_cal])
y_mix = np.concatenate([y_train_A, y_B_cal])

# 3. Covariate Shift Reduction (Re-Standardization)
# Fit scaler on the mixed distribution to align feature scales
scaler_mix = StandardScaler()
X_mix_scaled = scaler_mix.fit_transform(X_mix)
X_B_eval_scaled = scaler_mix.transform(X_B_eval)

# 4. Model Training (SVM)
print("[INFO] Training Support Vector Machine (RBF Kernel)...")
clf = SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42)
clf.fit(X_mix_scaled, y_mix)
print("[INFO] Training complete.")

# 5. Performance Evaluation
print(f"\n{'='*20} CLINICAL PERFORMANCE REPORT {'='*20}")

y_score = clf.predict_proba(X_B_eval_scaled)[:, 1]
ap = average_precision_score(y_B_eval, y_score)

# Calculate False Alarms per Hour (FA/h)
THRESHOLD = 0.11 # Empirically determined threshold
y_pred = (y_score >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_B_eval, y_pred, labels=[0, 1]).ravel()
total_hours = (len(y_B_eval) * EPOCH_LEN_S) / 3600.0
fa_per_hour = fp / total_hours

# Calculate Sensitivity at fixed False Positive Rate (FPR = 5%)
fprs, tprs, threshs = roc_curve(y_B_eval, y_score)
idx_fpr5 = np.where(fprs <= 0.05)[0][-1] # Index where FPR is closest to 0.05
recall_fpr5 = tprs[idx_fpr5]

print(f"1. Average Precision (AP):           {ap:.4f}")
print(f"   (Area under Precision-Recall Curve. >0.4 indicates strong performance on imbalanced data)")
print("-" * 60)
print(f"2. False Alarm Rate (FA/h):          {fa_per_hour:.2f} events/hour")
print(f"   (Threshold={THRESHOLD}. Clinical acceptance criteria is typically < 1.0/h)")
print("-" * 60)
print(f"3. Sensitivity @ FPR=5%:             {recall_fpr5:.4f} ({recall_fpr5*100:.1f}%)")
print(f"   (Recall capability when False Positive Rate is restricted to 5%)")
print("-" * 60)
print("\nDetailed Classification Report:")
print(classification_report(y_B_eval, y_pred))

[INFO] Initiating Transfer Learning Protocol...
[INFO] Performing Stratified Shuffle Split (20% Calibration / 80% Evaluation)...
   -> Calibration Set: 12681 samples (17 seizures)
   -> Evaluation Set:  50726 samples (70 seizures)
[INFO] Training Support Vector Machine (RBF Kernel)...
[INFO] Training complete.

==================== CLINICAL PERFORMANCE REPORT ====================
1. Average Precision (AP):           0.4270
   (Area under Precision-Recall Curve. >0.4 indicates strong performance on imbalanced data)
------------------------------------------------------------
2. False Alarm Rate (FA/h):          0.46 events/hour
   (Threshold=0.11. Clinical acceptance criteria is typically < 1.0/h)
------------------------------------------------------------
3. Sensitivity @ FPR=5%:             0.9857 (98.6%)
   (Recall capability when False Positive Rate is restricted to 5%)
------------------------------------------------------------

Detailed Classification Report:
              preci